This notebook introduces a minimal attention_based neural language learner trained on an artificial language with singular and plural number particles.

The experiment investigates whether the learner can generalise a distributional regularity to a noun-number combination absent from its training data.

Step 1: Define examples that belong to our artificial language

In [1]:
examples = ["one dax pa", "two dax ti"]

In [2]:
print(examples)

['one dax pa', 'two dax ti']


Step 2: Add examples of artificial language to the training data

In [3]:
examples.extend(["one mit pa", "two mit ti"])

In [4]:
print(examples)

['one dax pa', 'two dax ti', 'one mit pa', 'two mit ti']


Step 3: Add an example containing a new noun (singular only) to test generalisation later

In [5]:
examples.append("one wug pa")

In [6]:
print(examples)

['one dax pa', 'two dax ti', 'one mit pa', 'two mit ti', 'one wug pa']


Step 4: Count examples in the training data

In [7]:
len(examples)

5

Step 5: Split one example into individual words (tokens)

In [8]:
examples[1].split()

['two', 'dax', 'ti']

Step 6: Tokenize all training examples and display the result

In [9]:
for sentence in examples:
  print(sentence.split())

['one', 'dax', 'pa']
['two', 'dax', 'ti']
['one', 'mit', 'pa']
['two', 'mit', 'ti']
['one', 'wug', 'pa']


Step 7: Store the tokenized examples

In [10]:
tokens = []
for sentence in examples:
  tokens.append(sentence.split())

In [11]:
print(tokens)

[['one', 'dax', 'pa'], ['two', 'dax', 'ti'], ['one', 'mit', 'pa'], ['two', 'mit', 'ti'], ['one', 'wug', 'pa']]


Step 8: Collect all tokens into a single list (words)

In [12]:
words = []

for sentence in tokens:
  words.extend(sentence)

In [13]:
print(words)

['one', 'dax', 'pa', 'two', 'dax', 'ti', 'one', 'mit', 'pa', 'two', 'mit', 'ti', 'one', 'wug', 'pa']


Step 9: Create our vocabulary (unique tokens)


In [14]:
vocabulary = set(words)

In [15]:
print(vocabulary)

{'two', 'mit', 'wug', 'pa', 'dax', 'one', 'ti'}


In [16]:
vocabulary = sorted(vocabulary)

In [17]:
print(vocabulary)

['dax', 'mit', 'one', 'pa', 'ti', 'two', 'wug']


Step 10: Assign a unique numerical ID to each token


In [18]:
for number, word in enumerate(vocabulary):
  print(number, word)

0 dax
1 mit
2 one
3 pa
4 ti
5 two
6 wug


In [19]:
word_to_number = {}

for number, word in enumerate(vocabulary):
  word_to_number[word] = number

In [20]:
print(word_to_number)

{'dax': 0, 'mit': 1, 'one': 2, 'pa': 3, 'ti': 4, 'two': 5, 'wug': 6}


Step 11: Encode our training examples as sequences of token IDs

In [21]:
numbers = []

for word in tokens[0]:
  numbers.append(word_to_number[word])

In [22]:
print(numbers)

[2, 0, 3]


In [23]:
numerical_examples = []

for sentence in tokens:
  numbers = []
  for word in sentence:
    numbers.append(word_to_number[word])
  numerical_examples.append(numbers)

In [24]:
print(numerical_examples)

[[2, 0, 3], [5, 0, 4], [2, 1, 3], [5, 1, 4], [2, 6, 3]]


Step 12: Import PyTorch nad convert numerical examples into tensors

In [25]:
import torch
torch.manual_seed(42)

In [26]:
first_sentence = torch.tensor(numerical_examples[0])

In [27]:
print(first_sentence)

tensor([2, 0, 3])


In [28]:
tensors = []

for numbers in numerical_examples:
  tensors.append(torch.tensor(numbers))

In [29]:
print(tensors)

[tensor([2, 0, 3]), tensor([5, 0, 4]), tensor([2, 1, 3]), tensor([5, 1, 4]), tensor([2, 6, 3])]


Step 13: Create word embeddings (4 embedding dimensions per token)

In [30]:
embedding = torch.nn.Embedding(len(vocabulary), 4)

In [31]:
print(embedding(tensors[0]))

tensor([[-0.7521,  1.6487, -0.3925, -1.4036],
        [ 1.9269,  1.4873,  0.9007, -2.1055],
        [-1.1109,  0.0915, -2.3169, -0.2168]], grad_fn=<EmbeddingBackward0>)


In [32]:
print(embedding(tensors[0]).shape)

torch.Size([3, 4])


Step 14: Represent word order using positional embeddings

In [33]:
positions = torch.arange(3)
print(positions)

tensor([0, 1, 2])


In [34]:
position_embedding = torch.nn.Embedding(3, 4)

In [35]:
print(position_embedding(positions))

tensor([[-0.3278,  0.7950,  0.2815,  0.0562],
        [ 0.5227, -0.2384, -0.0499,  0.5263],
        [-0.0085,  0.7291,  0.1331,  0.8640]], grad_fn=<EmbeddingBackward0>)


In [36]:
combined = embedding(tensors[0]) + position_embedding(positions)

In [37]:
print(combined.shape)

torch.Size([3, 4])


Step 15: Prepare input for attention

In [38]:
input_batch = combined.unsqueeze(0)

In [39]:
print(input_batch.shape)

torch.Size([1, 3, 4])


Step 16: Add self-attention mechanism and mask (preventing access to future tokens)

In [40]:
attention = torch.nn.MultiheadAttention(embed_dim=4, num_heads=1, batch_first=True)

In [41]:
mask = torch.tensor([
    [False, True, True],
    [False, False, True],
    [False, False, False]
])

In [42]:
print(mask)

tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])


In [43]:
output, weights = attention(
    input_batch,
    input_batch,
    input_batch,
    attn_mask=mask
)

In [44]:
print(output.shape)

torch.Size([1, 3, 4])


In [45]:
print(weights)

tensor([[[1.0000, 0.0000, 0.0000],
         [0.4388, 0.5612, 0.0000],
         [0.4379, 0.2704, 0.2917]]], grad_fn=<MeanBackward1>)


Step 17: Use attention to predict the next word

In [46]:
prediction = torch.nn.Linear(4, len(vocabulary))

In [47]:
scores = prediction(output)

In [48]:
print(scores.shape)

torch.Size([1, 3, 7])


In [49]:
print(scores[0, 1, word_to_number["pa"]])
print(scores[0, 1, word_to_number["ti"]])

tensor(0.5728, grad_fn=<SelectBackward0>)
tensor(0.1698, grad_fn=<SelectBackward0>)


In [50]:
probabilities = torch.softmax(scores[0, 1], dim=0)

In [51]:
print(probabilities[word_to_number["pa"]])
print(probabilities[word_to_number["ti"]])

tensor(0.2321, grad_fn=<SelectBackward0>)
tensor(0.1552, grad_fn=<SelectBackward0>)


Step 18: Calculate the model's prediction error (loss)

In [52]:
correct_word = word_to_number["pa"]
loss = -torch.log(probabilities[correct_word])
print(loss.item())

1.460390567779541


In [53]:
loss.backward() # backpropagation

In [54]:
parameters = list(embedding.parameters())
parameters += list(position_embedding.parameters())
parameters += list(attention.parameters())
parameters += list(prediction.parameters())

In [55]:
optimizer = torch.optim.SGD(parameters, lr=0.1)

In [56]:
optimizer.step()

In [57]:
combined = embedding(tensors[0]) + position_embedding(positions)

input_batch = combined.unsqueeze(0)

output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask)

scores = prediction(output)

In [58]:
probabilities = torch.softmax(scores[0, 1], dim=0)

print(probabilities[word_to_number["pa"]].item())

0.27510425448417664


In [59]:
new_loss = -torch.log(probabilities[word_to_number["pa"]])

print(new_loss.item())

1.2906051874160767


Step 19: Prepare all examples for training

In [60]:
for sentence in tensors:
  print(sentence[:2], "->", sentence[2])

tensor([2, 0]) -> tensor(3)
tensor([5, 0]) -> tensor(4)
tensor([2, 1]) -> tensor(3)
tensor([5, 1]) -> tensor(4)
tensor([2, 6]) -> tensor(3)


In [61]:
for sentence in tensors:
  input_words = sentence[:2]
  combined = embedding(input_words) + position_embedding(positions[:2])
  print(combined.shape)

torch.Size([2, 4])
torch.Size([2, 4])
torch.Size([2, 4])
torch.Size([2, 4])
torch.Size([2, 4])


In [62]:
mask_2 = torch.tensor([
    [False, True],
    [False, False]
])

In [63]:
print(mask_2.shape)

torch.Size([2, 2])


In [64]:
for sentence in tensors:
  input_words = sentence[:2]
  combined = embedding(input_words) + position_embedding(positions[:2])
  input_batch = combined.unsqueeze(0)
  output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask_2)
  scores = prediction(output)
  print(scores[0, 1].shape)

torch.Size([7])
torch.Size([7])
torch.Size([7])
torch.Size([7])
torch.Size([7])


Step 20: Training the learner

In [65]:
print(scores[0, 1, word_to_number["pa"]])
print(scores[0, 1, word_to_number["ti"]])

tensor(0.7002, grad_fn=<SelectBackward0>)
tensor(0.6744, grad_fn=<SelectBackward0>)


In [66]:
for sentence in tensors:
  input_words = sentence[:2]
  combined = embedding(input_words) + position_embedding(positions[:2])
  input_batch = combined.unsqueeze(0)
  output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask_2)
  scores = prediction(output)
  print(scores[0, 1].shape)

  probabilities = torch.softmax(scores[0, 1], dim=0)
  loss = -torch.log(probabilities[sentence[2]])
  print(loss.item())

torch.Size([7])
1.2906051874160767
torch.Size([7])
1.8857005834579468
torch.Size([7])
1.5566015243530273
torch.Size([7])
1.9006749391555786
torch.Size([7])
1.5718485116958618


In [67]:
for epoch in range(10):
  total_loss = 0

  for sentence in tensors:
    optimizer.zero_grad()

    input_words = sentence[:2]
    combined = embedding(input_words) + position_embedding(positions[:2])
    input_batch = combined.unsqueeze(0)
    output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask_2)
    scores = prediction(output)

    probabilities = torch.softmax(scores[0, 1], dim=0)
    loss = -torch.log(probabilities[sentence[2]])

    loss.backward()
    optimizer.step()
    total_loss += loss.item()


  print("Epoch:", epoch + 1, "Average loss:", total_loss / len(tensors))

Epoch: 1 Average loss: 1.619532299041748
Epoch: 2 Average loss: 1.4065012216567994
Epoch: 3 Average loss: 1.2182567954063415
Epoch: 4 Average loss: 1.0501307547092438
Epoch: 5 Average loss: 0.9165469586849213
Epoch: 6 Average loss: 0.8117669522762299
Epoch: 7 Average loss: 0.716261449456215
Epoch: 8 Average loss: 0.6133146315813065
Epoch: 9 Average loss: 0.4956966616213322
Epoch: 10 Average loss: 0.36975878588855265


Step 21: Testing generalisation

In [68]:
test_words = torch.tensor([word_to_number["two"], word_to_number["wug"]])

In [69]:
print(test_words)

tensor([5, 6])


In [70]:
combined = embedding(test_words) + position_embedding(positions[:2])
input_batch = combined.unsqueeze(0)

In [71]:
print(input_batch.shape)

torch.Size([1, 2, 4])


In [72]:
output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask_2)
scores = prediction(output)

In [73]:
print(scores.shape)

torch.Size([1, 2, 7])


In [74]:
probabilities = torch.softmax(scores[0, 1], dim=0)

print("P(pa):", probabilities[word_to_number["pa"]].item())
print("P(ti):", probabilities[word_to_number["ti"]].item())

P(pa): 0.3984684646129608
P(ti): 0.4632337987422943


* The learner encountered "one dax pa" and "two dax ti"
* It encountered "one mit pa" and "two mit ti"
* It encountered "one wug pa"
* It never encountered two wug ti

After training, where does the model assign its probability following two wug, pa or ti? Is there evidence of generalisation beyond the specific examples in the training data?

Step 22: Final control test (a familiar example)

In [75]:
test_words = torch.tensor([word_to_number["one"], word_to_number["wug"]])

In [76]:
combined = embedding(test_words) + position_embedding(positions[:2])

input_batch = combined.unsqueeze(0)

output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask_2)

scores = prediction(output)

probabilities = torch.softmax(scores[0, 1], dim=0)

print("P(pa):", probabilities[word_to_number["pa"]].item())
print("P(ti):", probabilities[word_to_number["ti"]].item())

P(pa): 0.9234020709991455
P(ti): 0.06935063749551773
